In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# -----------------------------
# 1 Load Dataset
# -----------------------------
data = pd.read_csv("../data/processed_system_metrics.csv")

# -----------------------------
# 2 Drop timestamp
# -----------------------------
data = data.drop(columns=["timestamp"])

# -----------------------------
# 3 Create Future Prediction Target
# Predict CPU at next timestep
# -----------------------------
data["target_cpu"] = data["cpu_percent"].shift(-1)

# Remove last row (NaN target)
data = data.dropna()

# -----------------------------
# 4 Features and Target
# -----------------------------
X = data.drop(columns=["target_cpu"])
y = data["target_cpu"]

# -----------------------------
# 5 Train Test Split (80/20)
# Time-series safe split
# -----------------------------
split = int(len(data) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

# -----------------------------
# 6 Train Random Forest
# -----------------------------
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_model.fit(X_train, y_train)

# -----------------------------
# 7 Predictions
# -----------------------------
y_pred = rf_model.predict(X_test)

# -----------------------------
# 8 Evaluation
# -----------------------------
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Random Forest Results")
print("MAE:", mae)
print("RMSE:", rmse)

avg_cpu = y_test.mean()

accuracy = 100 - (mae / avg_cpu * 100)

print("Approx Accuracy:", accuracy)

# -----------------------------
# 9 Prediction Visualization
# -----------------------------
plt.figure(figsize=(10,5))

plt.plot(y_test.values, label="Actual CPU")
plt.plot(y_pred, label="Predicted CPU")

plt.legend()
plt.title("Random Forest CPU Prediction")
plt.xlabel("Time Step")
plt.ylabel("CPU %")

plt.show()

# -----------------------------
# 10 Feature Importance
# -----------------------------
importance = rf_model.feature_importances_

feature_importance = pd.Series(importance, index=X.columns)

plt.figure(figsize=(8,5))
feature_importance.sort_values().plot(kind="barh")

plt.title("Feature Importance (Random Forest)")
plt.xlabel("Importance")

plt.show()
